# ALADIN workflow
This Notebook follow the step-by-step workflow for HW-SW co-design with the proposed framework.

### Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from importlib import import_module
from pathlib import Path

p = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()


while not (p / "ALADIN").exists():  
    if p.parent == p:
        raise RuntimeError("Project root not found")
    p = p.parent
    
os.chdir(os.path.join(p, "ALADIN"))

MODEL_DIR = "./models"

MODELS = [
    "vgg",
    "resnet",
    "mobilenet_v1",
    "shufflenet",
    "efficientnet"
]

target_model = "efficientnet_case 3.onnx"
target_platform = 0


## 1. Implementation-Aware model

In [ ]:
import onnx
from qonnx.core.modelwrapper import ModelWrapper

model_path = os.path.join(MODEL_DIR, target_model)
mw = ModelWrapper(onnx.load(model_path))

In [ ]:
from qonnx.transformation.infer_shapes import InferShapes
from dory.Frontend_frameworks.QONNX.transformations.aladin_trace_model import ImplementationAwareTrace

mw = mw.transform(InferShapes()) 
mw.transform(ImplementationAwareTrace(
    output_path="./output", 
    file_name=f"STEP1_{target_model.split('.')[0]}",
    verbose=True
))

## 2. Code Generation with Dory

In [ ]:
FRONTEND = "QONNX"                  # "NEMO", "Quantlab", "QONNX"
HARDWARE = "PULP.PULP_gvsoc"        # "PULP.GAP8", "PULP.GAP8_L2", "PULP.PULP_gvsoc", "PULP.GAP9", "PULP.GAP9_NE16"
VERBOSE = "None"                    # "None", "Perf_final", "Check_all+Perf_final", "Last+Perf_final"
ISA = "mixed-sw"                    # "auto", "8bit", "mixed-hw", "mixed-sw"

dory_config = {
	"BNRelu_bits": 32,
	"code reserved space": 150000,
    "input_bits": 8,
    "input_signed": False
}

In [ ]:
meta_platforms = [
    {
        "num_cores": 8,
        "L1_capacity": 64000,
        "L2_capacity": 512000,
    },
    {
        "num_cores": 4,
        "L1_capacity": 64000,
        "L2_capacity": 320000,
    },
    {
        "num_cores": 2,
        "L1_capacity": 64000,
        "L2_capacity": 256000,
    }
]

In [ ]:
onnx_manager = import_module(f"dory.Frontend_frameworks.{FRONTEND}.Parser")
onnx_to_dory = onnx_manager.onnx_manager

graph = onnx_to_dory(
    model_path, 
    dory_config, 
    delta=16
).full_graph_parsing()

In [ ]:
from copy import deepcopy

PLATFORM = meta_platforms[target_platform]

NUM_CORES, L1_capacity, L2_capacity = PLATFORM.values()

onnx_manager = import_module(f'dory.Hardware_targets.{HARDWARE}.HW_Parser')
dory_to_dory_hw = onnx_manager.onnx_manager
graph_tmp = deepcopy(graph)
hw_graph = dory_to_dory_hw(
    graph_tmp,
    n_inputs=1,
    verify_checksum=False,
    L1_capacity=L1_capacity,
    L2_capacity=L2_capacity,
    config_file=dory_config,
    num_cores=NUM_CORES
).full_graph_parsing()

In [ ]:
onnx_manager = import_module(f'dory.Hardware_targets.{HARDWARE}.C_Parser')
dory_hw_to_c = onnx_manager.C_Parser
dory_hw_to_c(
    hw_graph, 
    dory_config,
    verbose_level=VERBOSE,
    perf_layer="Yes", 
    precision_library=ISA,
    app_directory="./application", 
    model_dir="./models",
    n_inputs=1,
    L1_capacity=L1_capacity,
    L2_capacity=L2_capacity,
    num_cores=NUM_CORES
).full_graph_parsing()

## 3. Platform-Aware modele

In [ ]:

measurement_path = os.path.join(
    "./output",
    f"STEP3_paltform {target_platform+1}_{target_model.split('.')[0]}.csv"
)

try:
    measured_df = pd.read_csv(measurement_path)
    measured_df = measured_df.dropna(subset=["layer_name", "num_cycles"])

    measured_df["layer_name"] = (
        measured_df["layer_name"]
        .astype(str)
        .str.strip()
    )

    measured_df["num_cycles"] = (
        pd.to_numeric(measured_df["num_cycles"], errors="coerce")
    )

    measured_df = measured_df.dropna(subset=["num_cycles"])
    measured_df["num_cycles"] = measured_df["num_cycles"].astype(int)
    measured_cycles = dict(
        zip(
            measured_df["layer_name"],
            measured_df["num_cycles"],
        )
    )
    
except:
    measured_cycles = None

measured_cycles

In [ ]:
import dory.Frontend_frameworks.QONNX.transformations.aladin_latency_model as alm



def rt_family(node, r):
    family = alm.classify_node(node, r)
    if str(getattr(node, "implementation", "")).lower() == "lut":
        return "lut"
    
    k = list(getattr(node, "kernel_shape", [1, 1]) or [1, 1])
    s = list(getattr(node, "strides", [1, 1]) or [1, 1])
    
    if family == "standard_conv":
        if k == [1, 1]:
            return "conv_1x1"
        if any(x > 1 for x in s):
            return "conv_spatial_downsample"
        return "conv_spatial"
    
    if family == "depthwise_conv":
        return (
            "depthwise_downsample"
            if any(x > 1 for x in s)
            else "depthwise"
        )
    
    return family



hardware_name = str(HARDWARE).split(".")[-1]

hw_spec_path = (
    Path("./dory")
    / "Hardware_targets"
    / "PULP"
    / hardware_name
    / "HW_description.json"
)

with hw_spec_path.open("r") as handle:
    raw_hw_spec = json.load(handle)

raw_hw_spec["memory"]["L1"]["dimension"] = int(
    L1_capacity
)
raw_hw_spec["memory"]["L2"]["dimension"] = int(
    L2_capacity
)

hw_spec = alm.prepare_pulp_hw_spec(
    raw_hw_spec
)


estimator = alm.LatencyEstimator.simple(
    hw_spec,
    num_cores=int(NUM_CORES),
    generated_code_dir=(
        "./application/DORY_network/src"
    ),
)

estimator.tiling = alm.TilingModelConfig(
    enabled=True,
    level_name="L1",
    compute_mode="full_layer",
    edge_tile_safety_factor=1.0,
    unknown_geometry_compute_factor=1.0,
    unknown_geometry_dma_redundancy_factor=1.05,
    max_dma_redundancy_factor=2.0,
    use_generated_tile_count_first=True,
)

raw_results = estimator.process(hw_graph)

if measured_cycles is None:
    results = raw_results
else:
    RT_RESERVE = 1.02
    ratios = {}

    for node, r in zip(hw_graph, raw_results):
        name = r["name"]

        if name not in measured_cycles:
            continue

        family = rt_family(node, r)

        ratio = float(measured_cycles[name]) / float(r["pessimistic_cycles"])
        ratios[family] = max(ratios.get(family, 0.0), ratio)


    rt_factors = {
        family: ratio * RT_RESERVE
        for family, ratio in ratios.items()
    }

    print("RT factors:")
    for family, factor in rt_factors.items():
        print(f"{family:25s} {factor:.3f}")


    results = []

    for node, r in zip(hw_graph, raw_results):
        result = dict(r)

        family = rt_family(node, r)
        factor = rt_factors.get(family, 1.0)

        result["family"] = family
        result["calibrated_pessimistic_cycles"] = int(
            np.ceil(
                r["pessimistic_cycles"] * factor
            )
        )

        results.append(result)


In [ ]:
df = pd.DataFrame([
    {
        "name": r["name"],
        "latency": alm.get_pessimistic_cycles(r),
    }
    for r in results
])
predicted_latency_path = f"./output/STEP2_{target_model.split('.')[0]}_platform {target_platform+1}_latency.csv"
df.to_csv(predicted_latency_path, index=False)

## 4. GVSoC simulation
*NOTE*: this portion of the notebooks run only inside the container.

In [ ]:
import subprocess


cmd =   f"""
        source .devcontainer/pulp_sdk.sh
        export PATH=/dory_env/bin:$PATH
        make -C ./application clean all run platform=gvsoc CORE={NUM_CORES}
        """
try:
    proc = subprocess.run(
        ["bash", "-c", cmd],
        check=True,
        capture_output=True,
        text=True,
        timeout=360,
    )
    
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"Build or run failed (exit {e.returncode}):\n"
        f"STDOUT:\n{e.stdout}\n"
        f"STDERR:\n{e.stderr}"
    )
except subprocess.TimeoutExpired as e:
    raise TimeoutError(
        f"Test timed out after 720s.\n"
        f"Partial STDOUT:\n{e.stdout}\n"
        f"STDERR:\n{e.stderr}"
    )
    
proc.stdout

## Check results:

In [ ]:
import csv

csv_path = os.path.join(
    "./output",
    f"STEP3_paltform {target_platform+1}_{target_model.split('.')[0]}.csv"
)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

In [ ]:
with open("./logs/HW_related/json_files/06_DORY_HW_tiled_graph.json", "r") as f:
    layers_tiling_info = json.load(f)["graph"]

In [ ]:
def sum_componenets(tile: dict) -> int:
    return sum((
        tile["weight_memory"],
        tile["bias_memory"],
        tile["constants_memory"],
        tile["input_activation_memory"],
        tile["output_activation_memory"],
        tile.get("lut_memory", 0),
    ))
                        

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "layer_name", 
        "MACs", 
        "num_cycles", 
        "MAC_per_cycle", 
        "num_cores", 
        "L1_mem", 
        "L2_mem", 
        "L1_tiling", 
        "L2_tiling"
    ])
    idx = 0
    for line in proc.stdout.splitlines():
        if not line.startswith("PERF_LOG"):
            continue
            
        info = line.strip().split(",")
        if len(info) != 6:
            continue
        
        _, layer_name, macs, cycles, perf, cores = info
        tiling_info = layers_tiling_info[idx]["Tiling_parameters"]
        idx += 1
        writer.writerow([
            layer_name,
            int(macs),
            int(cycles),
            float(perf),
            int(cores),
            L1_capacity,
            L2_capacity,
            sum_componenets(tiling_info["L1"]),
            sum_componenets(tiling_info["L2"])
        ])